In [1]:
from websocket import create_connection
import json
import pandas as pd
from datetime import date
from time import sleep

In [2]:
## web socket url
socket = 'wss://data.tradingview.com/socket.io/websocket'
# change session id based on current session value
session_id = 's_h1DU5odwE8df'
## example messages to be sent
# ~m~55~m~{"m":"chart_create_session","p":["cs_yDp6P0icM81I",""]}
# ~m~117~m~{"m":"resolve_symbol","p":["cs_yDp6P0icM81I","sds_sym_1","={\"adjustment\":\"splits\",\"symbol\":\"NSE:HDFCBANK\"}"]}
# ~m~82~m~{"m":"create_series","p":["cs_yDp6P0icM81I","sds_1","s1","sds_sym_1","1W",300,""]}

downloads = './data/nifty500_5yr_daily/'

In [4]:
# create message as per template and send
def create_msg(ws, fun, arg):
    ms = json.dumps({"m":fun,"p":arg})
    msg = f'~m~{len(ms)}~m~{ms}'
    ws.send(msg)

# format result data into pandas dataframe
def format_data(data):
    start = data.find('"s":[')
    end = data.find(',"ns":')
    trimmed = data[start+4 : end]
    final_data = [x["v"] for x in json.loads(data[start+4 : end])]
    # print(final_data)
    return final_data

In [5]:
def fetch_historic_data(symbol):
    # resolve_symbol argument
    sess_msg = [session_id,""]    
    resolve_symbol = '={\"adjustment\":\"splits\",\"symbol\":\"NSE:' + symbol + '\"}'
    res_sym_msg = [session_id,"sds_sym_1",resolve_symbol]
    ser_msg = [session_id,"sds_1","s1","sds_sym_1","1D",1030,""]
    # create fresh websocket connection
    ws = create_connection(socket)
    # send messages to websocket
    create_msg(ws, 'chart_create_session', sess_msg)
    create_msg(ws, 'resolve_symbol', res_sym_msg)
    create_msg(ws, 'create_series', ser_msg)
    
    while True :
        # receive response for the messages sent
        res = ws.recv()
        if "series_completed" in res:
            formatted_data = format_data(res)
            # convert json to pandas dataframe for analysis
            df_result = pd.DataFrame(formatted_data, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
            # convert epoch timestamp to date yyyy-mm-dd format
            df_result['date'] = df_result['date'].apply(lambda x : str(date.fromtimestamp(x)))
            df_result.sort_values('date', inplace=True)
            # calculate average
            df_result['avg'] = df_result.apply(lambda row : int((row['open'] + row['high'] + row['low'] + row['close'])/4), axis=1)
            # shutdown websocket connection for next iteration to work properly
            ws.shutdown()
            # break the infinite while loop
            break
    
    return df_result

In [6]:
# run for a sample symbol
df_result = fetch_historic_data('BAJAJ_AUTO')
df_result

,date,open,high,low,close,volume,avg
0,2021-01-13,3624.25,3633.0000,3577.0000,3600.8999,515154.0,3608
1,2021-01-14,3600.00,3658.6499,3545.0000,3576.0500,743848.0,3594
2,2021-01-15,3588.00,3623.0000,3541.0000,3580.3000,684649.0,3583
3,2021-01-18,3552.00,3618.0500,3505.0000,3563.7500,602004.0,3559
4,2021-01-19,3565.75,3664.0000,3561.1001,3641.0000,586410.0,3607
...,...,...,...,...,...,...,...
1025,2025-02-28,8234.75,8234.7500,7886.3000,7902.9000,580155.0,8064
1026,2025-03-03,7934.95,7998.3000,7679.6000,7714.8000,497052.0,7831
1027,2025-03-04,7652.10,7680.0000,7317.0500,7333.3000,1078953.0,7495
1028,2025-03-05,7332.20,7443.3000,7301.0000,7420.3000,749464.0,7374


In [8]:
# if loop breaks before completing all symbols, add the completed symbols to this list
completed_list = []

# use index/list of your choice
#/Users/abss/Desktop/Work/StockPrediction/share_analysis
nifty_500 = pd.read_csv('./data/ind_nifty500list.csv')
nifty_500

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,AIA Engineering Ltd.,Capital Goods,AIAENG,EQ,INE212H01026
...,...,...,...,...,...
495,Zee Entertainment Enterprises Ltd.,Media Entertainment & Publication,ZEEL,EQ,INE256A01028
496,Zensar Technolgies Ltd.,Information Technology,ZENSARTECH,EQ,INE520A01027
497,Zomato Ltd.,Consumer Services,ZOMATO,EQ,INE758T01015
498,Zydus Lifesciences Ltd.,Healthcare,ZYDUSLIFE,EQ,INE010B01027


In [9]:
symbols = list(sorted(set(nifty_500['Symbol']) - set(completed_list)))

for i, symbol in enumerate(symbols) :
    if '-' in symbol:
        symbol = symbol.replace('-', '_')
    df_result = fetch_historic_data(symbol)
    df_result.to_csv(downloads + symbol + '.csv', index=None)
    completed_list.append(symbol)
    sleep(1)
    print(i+1, symbol)

1 360ONE
2 3MINDIA
3 AADHARHFC
4 AARTIIND
5 AAVAS
6 ABB
7 ABBOTINDIA
8 ABCAPITAL
9 ABFRL
10 ABREL
11 ABSLAMC
12 ACC
13 ACE
14 ACI
15 ADANIENSOL
16 ADANIENT
17 ADANIGREEN
18 ADANIPORTS
19 ADANIPOWER
20 AEGISLOG
21 AFFLE
22 AIAENG
23 AJANTPHARM
24 AKUMS
25 ALKEM
26 ALKYLAMINE
27 ALOKINDS
28 AMBER
29 AMBUJACEM
30 ANANDRATHI
31 ANANTRAJ
32 ANGELONE
33 APARINDS
34 APLAPOLLO
35 APLLTD
36 APOLLOHOSP
37 APOLLOTYRE
38 APTUS
39 ARE&M
40 ASAHIINDIA
41 ASHOKLEY
42 ASIANPAINT
43 ASTERDM
44 ASTRAL
45 ASTRAZEN
46 ATGL
47 ATUL
48 AUBANK
49 AUROPHARMA
50 AVANTIFEED
51 AWL
52 AXISBANK
53 BAJAJ_AUTO
54 BAJAJFINSV
55 BAJAJHLDNG
56 BAJFINANCE
57 BALAMINES
58 BALKRISIND
59 BALRAMCHIN
60 BANDHANBNK
61 BANKBARODA
62 BANKINDIA
63 BASF
64 BATAINDIA
65 BAYERCROP
66 BBTC
67 BDL
68 BEL
69 BEML
70 BERGEPAINT
71 BHARATFORG
72 BHARTIARTL
73 BHARTIHEXA
74 BHEL
75 BIKAJI
76 BIOCON
77 BIRLACORPN
78 BLS
79 BLUEDART
80 BLUESTARCO
81 BOSCHLTD
82 BPCL
83 BRIGADE
84 BRITANNIA
85 BSE
86 BSOFT
87 CAMPUS
88 CAMS
89 CANBK
90 CAN